# 04 — Orchestrated Workflow

**Goal:** combine many small agents into a larger workflow.

orqest gives you four primitives for combining agents. Each one does **one thing**, so you can mix and match them. Here is what they do, in plain words:

| Primitive | What it does |
|---|---|
| `Pipeline` | Run steps **in order**. Each step's output feeds the next. |
| `Parallel` | Run steps **at the same time**, then merge the results. |
| `Router` | Pick **one** path based on the input. |
| `RefinementLoop` | Run a step **again and again** until an evaluator is happy. |

They compose. A `Router` can route into a `Pipeline`. A `Pipeline` step can be a `Parallel`. We will build a small research-brief workflow that nests several of them, then look at what just happened using the built-in observability.

**Before you start:** you need a `.env` at the repo root with `LLM_API_KEY` and `LLM_MODEL`.


## 1. Load the config

In [1]:
from orqest import load_config

config = load_config()
print(f"Model: {config.llm_model}")

Model: openai:gpt-5.2


## 2. The cast — a few tiny agents

We need a handful of small agents to put into the workflow. They all share one simple output type (`Text`) and are told to be brief. The point is the **flow**, not the prose.

The `make_agent` helper just keeps the rest of the notebook clean. It builds a fresh `BaseAgent` with the given name and role.


In [2]:
from pydantic import BaseModel
from orqest.agents import BaseAgent, GlobalState


class Text(BaseModel):
    text: str


def make_agent(name: str, role: str) -> BaseAgent:
    """Build a tiny single-output agent with a brief-by-instruction prompt."""

    class _Agent(BaseAgent[GlobalState, Text]):
        async def _run_implementation(self, state: GlobalState, **kwargs) -> Text:
            result = await self.call_model(state.get_latest_message("user"), state)
            return result.output

    return _Agent(
        agent_name=name,
        system_prompt=f"{role} Respond in 2-3 sentences, no preamble.",
        output_type=Text,
        model=config.llm_model,
        api_key=config.llm_api_key,
    )


tech_agent      = make_agent("tech_angle",      "You explain the *technical* angle of a topic.")
practical_agent = make_agent("practical_angle", "You explain the *practical, day-to-day* angle of a topic.")
synthesizer     = make_agent("synthesizer",     "You merge the angles you are given into one tight brief.")
refiner         = make_agent("refiner",         "You tighten prose, preserving the key point.")
quick_agent     = make_agent("quick",           "You give a one-paragraph quick take on a topic.")

print("5 agents ready.")

5 agents ready.


## 3. `Parallel` — fan out and merge

`Parallel` runs each step at the same time. When all of them finish, the merge strategy decides what the final output looks like. `collect_all` gives you a list of every output. There is also `first_wins`, or you can pass your own callable.

Why use it? Two reasons:

- **Speed.** Two LLM calls that take 5 seconds each finish in 5 seconds instead of 10.
- **Different viewpoints.** Two agents with different prompts give you two angles on the same topic.

Here, two agents look at the same topic from different angles, at the same time.


In [3]:
from orqest.orchestration import Parallel, MergeStrategy

TOPIC = "vector databases"

fanout = Parallel([tech_agent, practical_agent], merge=MergeStrategy.collect_all)
par_result = await fanout.run(TOPIC)

for angle in par_result.outputs:
    print(f"- {angle.text}\n")

- Vector databases store and index high-dimensional embedding vectors (plus metadata) to support fast approximate nearest-neighbor search using structures like HNSW graphs, IVF/IVF-PQ, or LSH, typically under cosine/dot/L2 metrics. They’re used for semantic retrieval (RAG), similarity search, and recommendations by embedding inputs with an ML model, querying k-NN, and optionally applying metadata filters and re-ranking.

- Vector databases store data as embeddings (number vectors) so you can quickly retrieve “most similar” items—useful day-to-day for semantic search, recommendations, and RAG chatbots that need to pull relevant docs by meaning, not keywords. In practice you generate embeddings for your text/images, upsert them with metadata, then query with a new embedding to get top‑K matches and filter by metadata (date, user, product) before sending results to your app/LLM.



## 4. `RefinementLoop` — try, evaluate, try again

A `RefinementLoop` repeats a step until an evaluator says it's good enough (or `max_iterations` is hit).

It needs three things:

- **a step** — the agent that does the work,
- **an evaluator** — a function that grades the output and returns `EvalResult(passed=..., feedback=..., score=...)`,
- **a state_updater** — builds the next input from the current output and the evaluator's feedback.

The pattern is useful for *quality* loops: tighten prose, fix code that fails a linter, hit a word limit. In this example we ask the `refiner` to shorten a long paragraph until it is under 45 words.


First, the wordy input and the evaluator + state-updater functions.

In [ ]:
from orqest.orchestration import RefinementLoop
from orqest.orchestration.loop import EvalResult

WORDY = (
    "Vector databases are, in essence, a specialised kind of database that has been "
    "designed and optimised specifically for the storage and the retrieval of "
    "high-dimensional vector embeddings, which are the numerical representations "
    "that machine learning models produce, and they support similarity search which "
    "is the operation of finding the stored vectors that are closest to a given "
    "query vector."
)


def evaluator(output: Text) -> EvalResult:
    """Return passed=True when the text is short enough."""
    words = len(output.text.split())
    return EvalResult(passed=words <= 45, feedback=f"{words} words", score=float(words))


def state_updater(current_input: str, output: Text, ev: EvalResult) -> str:
    """Build the next prompt: feed back the current draft + the evaluator's note."""
    return f"Tighten this to under 45 words ({ev.feedback} now):\n\n{output.text}"


print("Evaluator and state-updater defined.")

Now run the loop. We bound it to 3 iterations as a safety net.

In [4]:
loop = RefinementLoop(refiner, evaluator, state_updater=state_updater, max_iterations=3)
loop_result = await loop.run(WORDY)

print(f"exit_reason: {loop_result.exit_reason}  ({loop_result.iterations} iterations)")
for rec in loop_result.history:
    print(f"  iter {rec.iteration}: {rec.eval_result.feedback}  passed={rec.eval_result.passed}")

print(f"\nfinal: {loop_result.output.text}")

exit_reason: passed  (1 iterations)
  iter 1: 29 words  passed=True

final: Vector databases are specialised systems for storing and retrieving high‑dimensional vector embeddings produced by ML models. They’re optimised for similarity search—finding stored vectors closest to a given query vector.


## 5. `Pipeline` — chain the stages

A `Pipeline` runs steps in order. Each step's output becomes the next step's input.

There is one tiny detail to know: every primitive returns its own result type. `Parallel.run` returns a `ParallelResult`, not a string. So if we want to put a `Parallel` *inside* a `Pipeline`, we need a small adapter function that runs the `Parallel` and reshapes its output into whatever the next step expects.

That is what `fan_out_stage` does below. It is a normal `async def` — pipelines accept three step types: agents, callables, and proper `Step` objects. Each gets wrapped automatically.

**Bonus feature:** `run_stream` yields events at every lifecycle point. You get built-in observability for free, without setting up tracing yourself.


First, define the adapter and assemble the pipeline.

In [ ]:
from orqest.orchestration import Pipeline


async def fan_out_stage(topic: str) -> str:
    """Stage 1: fan out into two angles, then format them for the synthesizer."""
    result = await Parallel([tech_agent, practical_agent]).run(topic)
    tech, practical = result.outputs
    return (
        f"TECHNICAL ANGLE: {tech.text}\n\n"
        f"PRACTICAL ANGLE: {practical.text}\n\n"
        "Merge these into one tight brief."
    )


pipeline = Pipeline([fan_out_stage, synthesizer], name="research_brief")
print("Pipeline ready.")

Now run it with `run_stream` and watch the lifecycle events. The final brief is in the last event's data.

In [5]:
events = []
async for ev in pipeline.run_stream(TOPIC):
    events.append(ev)
    print(f"{ev.event_type:18s} step={ev.step_name}")

brief = events[-1].data["output"]
print(f"\nbrief: {brief.text}")

pipeline_start     step=
step_start         step=fan_out_stage


step_complete      step=fan_out_stage
step_start         step=synthesizer


step_complete      step=synthesizer
pipeline_complete  step=

brief: A vector database stores high‑dimensional embedding vectors (with metadata and CRUD/upsert) and enables fast approximate nearest‑neighbor similarity search via indexes like HNSW or IVF/PQ using cosine/dot/L2 metrics. Practically, you embed items, upsert the vectors plus metadata, then query with a new embedding—optionally combining keyword/metadata filters for hybrid retrieval—to quickly return semantically similar results at million‑scale for semantic search, recommendations, and RAG chatbots.


## 6. `Router` — pick the path

A `Router` checks each route's condition in order. The first match wins. If nothing matches, the `fallback` runs.

Here the router checks whether the user asked for a "deep" brief. If yes, it sends the request through the full `Pipeline` we just built. If not, it sends the request to the small `quick_agent`.

We will also wrap each run in a tracer span. Orchestration primitives don't fire the hook lifecycle (that's for `CompoundTool` / `SubAgentTool`), so the workbench's tracer is the right place to measure them.


Build the router and the workbench.

In [ ]:
import tempfile, os
from orqest.orchestration import Router, Route
from orqest.workbench import Workbench
from orqest.memory import LocalMemoryStore


async def deep_route(request: str):
    return await pipeline.run(request)


router = Router(
    [Route("deep", deep_route, condition=lambda r: "deep" in r.lower())],
    fallback=quick_agent,
    name="brief_router",
)

wb = Workbench(memory=LocalMemoryStore(os.path.join(tempfile.mkdtemp(), "wf.db")))
print("Router and workbench ready.")

Now send two requests through it: one with "deep", one without. Each call is bracketed in a tracer span so we can measure it afterwards.

In [6]:
span = wb.tracer.start_span("deep_request", agent_name="brief_router")
deep_out = await router.run("Please do a deep brief on vector databases.")
wb.tracer.end_span(span, status="ok")

span = wb.tracer.start_span("quick_request", agent_name="brief_router")
quick_out = await router.run("Quick take on vector databases?")
wb.tracer.end_span(span, status="ok")

print(f"deep  -> {deep_out.text[:110]}")
print(f"quick -> {quick_out.text[:110]}")

deep  -> A vector database stores high‑dimensional embedding vectors and uses approximate nearest‑neighbor (ANN) indexe
quick -> Vector databases store and index high‑dimensional embeddings so you can do fast similarity search (nearest nei


The tracer recorded one span per request. Let's read them back — this is what a real observability backend (Jaeger, Tempo, etc.) would receive.

In [7]:
for s in wb.tracer.export_json():
    print(f"{s['name']:16s} {s['status']:4s} {s['duration_ms']:.0f} ms")

deep_request     ok   7531 ms
quick_request    ok   3487 ms


## Recap

Here is the full picture:

1. **`Parallel`** runs N steps at the same time and merges the outputs. Use it for speed and for multi-angle thinking.
2. **`RefinementLoop`** runs one step in a try-evaluate-try loop. Use it for quality gates: word counts, lint, type check, anything that has a clear pass/fail.
3. **`Pipeline`** chains steps in order. Each step feeds the next.
4. **`Router`** picks one path based on the input. Use it for "easy questions go to a cheap agent, hard ones go to the full pipeline".

**Two things to remember:**

- The primitives accept three kinds of steps: a `BaseAgent`, a plain `async def`, or a proper `Step` object. They all work side by side because of auto-coercion.
- Each primitive has its **own** result type. When you nest one inside another, a small adapter function bridges the two shapes. This is composition, not inheritance — and it stays simple even when the workflow grows.

**One observability tip:** orchestration primitives don't run the hook lifecycle. To measure them, use the workbench's tracer (`start_span` / `end_span`). For agent-tool interactions, hooks fire automatically.
